In [275]:
import networkx as nx
import numpy as np
import scipy.sparse as sp
import os
import time
from collections import defaultdict
# import tensorflow as tf
import pandas as pd
import pdb
from ast import literal_eval
import copy
import pickle
import sys
sys.path.append(os.path.join(os.path.dirname(sys.path[0]),'sim'))
from params import ENVIRONMENT_BOUNDARY_X, TIME_LIMIT, COMMIT_THRESHOLD
from math import sqrt
# from itertools import compress
from concurrent.futures import ThreadPoolExecutor, as_completed
import csv

In [276]:

STATES = {'RECRUIT':0.0/6.0, 'ASSESS':1.0/6.0, 'TRAVEL_HOME_TO_RECRUIT':2.0/6.0, 'TRAVEL_SITE':3.0/6.0, 
          'OBSERVE':4.0/6.0, 'EXPLORE':5.0/6.0, 'TRAVEL_HOME_TO_OBSERVE':6.0/6.0}

MAX_DIST=ENVIRONMENT_BOUNDARY_X[-1]


In [277]:

def getSiteID(site):
    if site == 'H':
        return -1
    else:
        try:
           return site
        except:
            pdb.set_trace()

def node_to_color_black(node):
    if node[3] == 0.0:
        return True
    else:
        return False

def node_to_color_green(node, quals):
    dancers = [(1, q) for a, q in zip(node[0::4], node[3::4]) if a == 0.0]
    q = 0
    # pdb.set_trace()
    for d in dancers:
        if d[1] == np.max(quals):
            q += 1
    
    if q > 5:
        return True
    else:
        return False
    
def node_to_color_red(node, quals):
    dancers = [(1, q) for a, q in zip(node[0::4], node[3::4]) if a == 0.0]
    q = 0
    # pdb.set_trace()
    for d in dancers:
        if d[1] != np.max(quals):
            q += 1
    if q > 5:
        return True
    else:
        return False


def round_function(x, d):
    new = []
    for r in x:
        new.append(np.round(r,decimals=d))
    # pdb.set_trace()
    return tuple(new)


def parse_row(r, quals, poses):
    
    # agent_states = literal_eval(r[4])
    # agent_sites = literal_eval(r[-2])
    # agent_positions = literal_eval(r[2])
    # pdb.set_trace()
    # node = literal_eval(r[-1])
    # pdb.set_trace()
    node = literal_eval(r[1].node_new)

    # node = get_current_state(agent_states, agent_sites, agent_positions, poses, quals)
    return node

In [278]:
def process_file(fileName, site_conv, time_conv, entry, folder, Gtime, Gsucc, Gedge):

    # While line = F.readline():
    # 	p_line = parse(line)	

    # 	node = tuple(p_line(4), p_line(6), …. ) // some function

    # 	//if node not in N:
    # 	//	N[node] = indCounter
    # 	//	indCounter++

    # 	if PrevNode != nil:
    # 		G[PrevNode][node]++

    # // convert G to tensor
    quals = entry[1].iloc[0]
    poses = entry[1].iloc[1]
    # prev = time.time()*1000.0
    # print(site_conv)
    success_now = site_conv/max(quals)      
    # counter = 0
    file = pd.read_csv(folder+fileName)
    for row in file.iterrows():
        # pdb.set_trace()
        node = parse_row(row, quals, poses)
        #pdb.set_trace()
        # if node not in G:
        

        Gtime[node].append(time_conv - row[1].time)
        Gsucc[node].append(success_now)
            # G[node]
        # if node not in Gedge:
        #     Gedge[node] = defaultdict(int)

        # if prev_node is not None:
        #     Gedge[prev_node][node] += 1
            
        # counter += 1
        # prev_node = node

    return Gtime, Gsucc, Gedge

In [279]:
def main():
    folder_main = './data/1000_len_sims/'
    dataset = 'test'
    folder = folder_main + dataset + '/'
    files = os.listdir(folder)
    files = [file for file in files if file.endswith('.csv') and file.startswith('1')]
    files = np.sort(files)
    # data_files = []
    metadata_file = folder + 'metadata.csv'
    # folder_graph = './graphs/RS/fast_multiple_graphs/'
    # new_metadata_file = folder_graph + 'metadata.csv'
    # graph_metaFile = 'graphMetadata.csv'
    # meta_arr = []
    metadata = pd.read_csv(metadata_file) 
    metadata.site_qualities=metadata.site_qualities.apply(literal_eval)
    metadata.site_positions=metadata.site_positions.apply(literal_eval)
    metadata.site_positions=metadata.site_positions.apply(lambda x: tuple([tuple(a) for a in x]))
    metadata.site_qualities=metadata.site_qualities.apply(lambda x: tuple(x))
    df = metadata.groupby(by=['site_qualities', 'site_positions', 'num_agents'], as_index=False).agg(lambda x: x.tolist())
    Gtime = defaultdict(list)
    Gsucc = defaultdict(list)
    Gedge = defaultdict(defaultdict)
    for some_id, entry in enumerate(df.iterrows()):

        print("time now 1: 0")
        print(entry)
        # pdb.set_trace()
        # list_of_dicts = []
        # fileName, site_conv, time_conv, entry, folder, Gtime, Gsucc
        files_to_process = [(fileName, site_conv, time_conv, entry, folder, Gtime, Gsucc, Gedge) 
                            for fileName, site_conv, time_conv in zip(entry[1].iloc[4], entry[1].iloc[6], entry[1].iloc[7])]
        
        # files_to_process = [(fileName, site_conv, time_conv, entry, folder, folder_graph1, folder_graph2) 
        #                     for fileName, site_conv, time_conv in zip(entry[1].iloc[4], entry[1].iloc[6], entry[1].iloc[7])]
        # with ThreadPoolExecutor(max_workers=8) as executor:
        #     futures = [executor.submit(process_file, *file_info) for file_info in files_to_process]
            
        #     for future in as_completed(futures):
        #         try:
        #             Gtime, Gsucc = future.result()
        #             # list_of_dicts.append(result)
        #             # print(f"{result}: {fileName}")
        #         except Exception as exc:
        #             print(f"File generated an exception: {exc}")

        for fileinfo in files_to_process:

            Gtime, Gsucc, Gedge = process_file(fileinfo[0], fileinfo[1], fileinfo[2], fileinfo[3], fileinfo[4], fileinfo[5], fileinfo[6], fileinfo[7])
            # print('processed file: ', fileinfo[0])
            # list_of_dicts.append(result)
        #     break
        # break
    # np.save()
    # pdb.set_trace()
    # newfname =  str(entry[1].iloc[0]) + str(entry[1].iloc[1]) + str(entry[1].iloc[2])
    # fname = newfname + '_' + '_noAgentPos_single_sim' + '.pickle'
    fil =  open(folder_main + 'graphs/all'+dataset+'time.pickle', 'wb')
    pickle.dump(Gtime, fil)   
    fil.close() 
    fil =  open(folder_main + 'graphs/all'+dataset+'succ.pickle', 'wb')
    pickle.dump(Gsucc, fil)   
    fil.close() 
    # fil =  open('alledges', 'wb')
    # pickle.dump(Gedge, fil)   
    # fil.close() 

In [280]:
if __name__ == "__main__":
    main()


time now 1: 0
(0, site_qualities                           (0.156, 0.906, 0.87, 0.95)
site_positions    ((150.0, 0.0), (0.0, 150.0), (-150.0, 0.0), (-...
num_agents                                                       10
Unnamed: 0        [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...
file_name         [1721021195573711.csv, 1721021195752015.csv, 1...
hub_position      [[0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0...
site_converged    [0.87, 0.87, 0.87, 0.87, 0.87, 0.87, 0.906, 0....
time_converged    [84, 9, 139, 23, 47, 147, 573, 53, 90, 189, 12...
start_state       [(('OBSERVE', None), ('TRAVEL_SITE', 2), ('EXP...
maxTime           [20000, 20000, 20000, 20000, 20000, 20000, 200...
timelimitsave     [20000, 20000, 20000, 20000, 20000, 20000, 200...
Name: 0, dtype: object)


In [281]:
# folder = './data/quorumsims/train/'
# files = os.listdir(folder)
# files = [file for file in files if file.endswith('.csv') and file.startswith('1')]
# files = np.sort(files)
# # data_files = []
# metadata_file = folder + 'metadata.csv'
# # folder_graph = './graphs/RS/fast_multiple_graphs/'
# # new_metadata_file = folder_graph + 'metadata.csv'
# # graph_metaFile = 'graphMetadata.csv'
# # meta_arr = []
# metadata = pd.read_csv(metadata_file) 
# metadata.site_qualities=metadata.site_qualities.apply(literal_eval)
# metadata.site_positions=metadata.site_positions.apply(literal_eval)
# metadata.site_positions=metadata.site_positions.apply(lambda x: tuple([tuple(a) for a in x]))
# metadata.site_qualities=metadata.site_qualities.apply(lambda x: tuple(x))
# df = metadata.groupby(by=['site_qualities', 'site_positions', 'num_agents'], as_index=False).agg(lambda x: x.tolist())
# Gtime = defaultdict(list)
# Gsucc = defaultdict(list)
# Gedge = defaultdict(defaultdict)
# for some_id, entry in enumerate(df.iterrows()):
#     break
# # print("time now 1: 0")
# print(entry)
# # pdb.set_trace()
# # list_of_dicts = []
# # fileName, site_conv, time_conv, entry, folder, Gtime, Gsucc
# files_to_process = [(fileName, site_conv, time_conv, entry, folder, Gtime, Gsucc, Gedge) 
#                         for fileName, site_conv, time_conv in zip(entry[1].iloc[4], entry[1].iloc[6], entry[1].iloc[7])]



In [282]:
# file1 = files_to_process[0]
# fileName, site_conv, time_conv, entry, folder, Gtime, Gsucc, Gedge = file1

In [283]:
# file1

In [284]:
# # time_conv

# # // convert G to tensor
# quals = entry[1].iloc[0]
# poses = entry[1].iloc[1]
# print(site_conv)
# success_now = site_conv/max(quals)



In [285]:

# # fileName, site_conv, time_conv, entry, folder, Gtime, Gsucc, Gedge

# counter = 0
# file = pd.read_csv(folder+fileName)
# for row in file.iterrows():
#         break
# # pdb.set_trace()
# # node = parse_row(row, quals, poses)
# node = literal_eval(row[1].node)
# #pdb.set_trace()
# # if node not in G:
# print(node)


In [286]:

# # Gtime[node].append(time_conv - counter)
# # Gsucc[node].append(success_now)
#         # G[node]
# if node not in Gedge:
#         Gedge[node] = defaultdict(int)

# if prev_node is not None:
#         Gedge[prev_node][node] += 1
        
# counter += 1
# prev_node = node

In [287]:
# # graph_metaFile = 'graphMetadata.csv'
# f = open('graphMetadata'+'succ.pickle.csv', 'rb')
# g = pickle.load(f) 
# f.close()  

In [288]:
# # os.listdir('./')
# fnew = open('nodessucc.pickle', 'wb')
# pickle.dump(g, fnew)
# fnew.close()

In [289]:
# for node in g:
#     print(node, g[node])
#     break